# nb54 - Metric-aligned training losses: trimmed pinball and coverage-width objective (H18)

**Error analysis.** The record recipe trains pinball on ALL events, but sigma_eff ignores the worst 32% by construction - training pays loss on tails the metric never sees (nb53: the high-contamination tail is unfixable without labels, yet it still shapes the gradients).

**Question.** Do losses that mirror the metric's trimming/width structure beat plain pinball?

**Hypotheses.** H18a trimmed: drop the largest 25% per-batch pinball losses (arXiv:2308.02293 reports high trim rates best). H18b coverage-width: minimize the mean q25-q75 interval width subject to a soft 50%-coverage penalty (Pearce et al., ICML 2018, arXiv:1802.07167) added to pinball. Verified gap: no published smooth sigma_eff surrogate exists - a win here is standalone novelty.

**Proof criterion.** Recipe identical to nb52 EMA (quant + clean-aux + EMA 0.999); 2 seeds per loss; anchor = nb52 EMA singles 0.0424 +/- 0.0003. Win = >0.002 overall or in any E>17 bin. Hyperparameters fixed a priori: trim rate 0.25; lambda_qd 0.5, tau 0.02, target coverage 0.5. No scans.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import SubNetFQ, QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB54_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB54_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 136s


In [2]:
TRIM = 0.25
LAM_QD = 0.5
TAU = 0.02
def loss_fn(q, yb, config):
    d = yb - q
    pin = torch.maximum(QS * d, (QS - 1) * d).mean(1)
    if config == 'trim':
        k = max(1, int((1.0 - TRIM) * pin.numel()))
        return torch.topk(pin, k, largest=False).values.mean()
    if config == 'qd':
        width = (q[:, 2] - q[:, 1]).abs() + (q[:, 1] - q[:, 0]).abs()
        inside = torch.sigmoid((yb.squeeze(1) - q[:, 0]) / TAU) * torch.sigmoid((q[:, 2] - yb.squeeze(1)) / TAU)
        cov_pen = torch.relu(0.5 - inside.mean()) ** 2
        return pin.mean() + LAM_QD * (width.mean() + 10.0 * cov_pen)
    return pin.mean()
def train_eval(config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb54_{config}_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m, b): return m(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m):
        m.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            loss_fn(fwd(model, b), T['Y'][b], config).backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [3]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
JOBS = {'smoke': [('trim', 0), ('qd', 0)],
        'full': [(cfg, s) for cfg in ('trim', 'qd') for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb54_losses{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    t1 = time.time()
    sig, pe = train_eval(config, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb54_pred{TAG}_{config}_s{seed}.npy', pe)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

resume, done: [('trim', 0), ('trim', 1)]
skip trim 0
skip trim 1


  resume qd s0 from epoch 80


qd seed 0: sigma_eff 0.0418 (187s)


qd seed 1: sigma_eff 0.0418 (2032s)


config  seed  sigma_eff  elapsed
  trim     0     0.0747      474
  trim     1     0.0629      757
    qd     0     0.0418      187
    qd     1     0.0418     2032


## Verdict vs the EMA anchor

Anchor: nb52 EMA singles 0.0424 +/- 0.0003 (identical recipe, plain pinball). Win = >0.002 overall or in any E>17 bin.

In [4]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
print('anchor: nb52 EMA singles 0.0424 +/- 0.0003 | stack record 0.0409')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for cfg in ('trim', 'qd'):
    preds = [np.load(OUT / f'nb54_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb54_pred{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'{cfg:5s} mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('      per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))

anchor: nb52 EMA singles 0.0424 +/- 0.0003 | stack record 0.0409
trim  mean 0.0688 +/- 0.0059 | ens 0.0666
      per-bin 0.1023 / 0.0602 / 0.0505 / 0.0470 / 0.0519 / 0.0544
qd    mean 0.0418 +/- 0.0000 | ens 0.0413
      per-bin 0.0650 / 0.0473 / 0.0344 / 0.0347 / 0.0343 / 0.0344
